In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_DIR = Path("data/processed")

cpi = pd.read_csv(DATA_DIR / "canada_cpi_cleaned.csv")
gdp = pd.read_csv(DATA_DIR / "canada_gdp_cleaned.csv")
labour = pd.read_csv(DATA_DIR / "canada_labour_market_cleaned.csv")
bcpi = pd.read_csv(DATA_DIR / "canada_bcpi_cleaned.csv")
bank = pd.read_csv(DATA_DIR / "canada_bank_of_canada_cleaned.csv")
stocks = pd.read_csv(DATA_DIR / "canada_stocks_cleaned.csv")

# Convert dates
for df in [cpi, gdp, labour, bcpi, bank, stocks]:
    df["Date"] = pd.to_datetime(df["Date"])

###  Standardizing BCPI column names.

In [3]:
bcpi = bcpi.rename(columns={
    "M.BCPI": "BCPI_Total",
    "M.BCNE": "BCPI_NonEnergy",
    "M.ENER": "BCPI_Energy",
    "M.MTLS": "BCPI_Metals",
    "M.FOPR": "BCPI_Forestry",
    "M.AGRI": "BCPI_Agriculture",
    "M.FISH": "BCPI_Fish"
})

### Standardizing Labour Market column names

In [4]:
labour = labour.rename(columns={
    "Labour force": "Labour_Force",
    "Participation rate": "Participation_Rate",
    "Unemployment rate": "Unemployment_Rate"
})

### Outer Merge

In [5]:
monthly = (
    cpi
    .merge(gdp, on="Date", how="outer")
    .merge(labour, on="Date", how="outer")
    .merge(bcpi, on="Date", how="outer")
)

monthly = monthly.sort_values("Date").reset_index(drop=True)

print("Shape:", monthly.shape)
print("Start:", monthly["Date"].min())
print("End:", monthly["Date"].max())

display(monthly.head())
display(monthly.tail())

Shape: (1354, 17)
Start: 1914-01-01 00:00:00
End: 2026-10-01 00:00:00


,Date,CPI,Inflation_YoY,GDP_Real_Millions_CAD,GDP_MoM,GDP_YoY,Employment,Labour_Force,Participation_Rate,Unemployment_Rate,BCPI_Total,BCPI_NonEnergy,BCPI_Energy,BCPI_Metals,BCPI_Forestry,BCPI_Agriculture,BCPI_Fish
0,1914-01-01,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1914-02-01,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1914-03-01,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1914-04-01,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1914-05-01,5.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Date,CPI,Inflation_YoY,GDP_Real_Millions_CAD,GDP_MoM,GDP_YoY,Employment,Labour_Force,Participation_Rate,Unemployment_Rate,BCPI_Total,BCPI_NonEnergy,BCPI_Energy,BCPI_Metals,BCPI_Forestry,BCPI_Agriculture,BCPI_Fish
1349,2026-06-01,169.0,2.798054,2371177.0,0.351224,2.060293,21139.7,22608.9,65.0,6.5,715.35,575.40,1503.34,1161.52,476.36,354.47,1670.00
1350,2026-07-01,169.9,3.032141,2371788.0,0.025768,1.424995,21214.8,22669.4,65.1,6.4,697.96,574.74,1432.17,1118.44,490.89,365.57,1682.41
1351,2026-08-01,169.8,3.033981,2371635.0,-0.006451,1.483505,21173.1,22632.6,65.0,6.4,709.81,579.20,1468.40,1165.55,474.35,359.83,1664.17
1352,2026-09-01,170.0,3.092784,NaN,NaN,NaN,21104.8,22579.7,64.8,6.5,747.71,576.51,1636.53,1148.65,471.24,361.77,1730.27
1353,2026-10-01,NaN,NaN,NaN,NaN,NaN,21100.1,22607.7,64.8,6.7,731.58,557.51,1620.08,1103.58,458.62,351.04,1730.27


In [6]:
monthly.to_csv(
    DATA_DIR / "canada_macro_monthly_combined.csv",
    index=False
)

In [7]:
monthly["Availability_Date"] = (
    monthly["Date"] + pd.DateOffset(months=2)
)

monthly = monthly.sort_values("Availability_Date")

In [8]:
print(monthly.shape)
print(monthly.head())
print(monthly.tail())

(1354, 18)
        Date  CPI  Inflation_YoY  GDP_Real_Millions_CAD  GDP_MoM  GDP_YoY  \
0 1914-01-01  6.0            NaN                    NaN      NaN      NaN   
1 1914-02-01  6.0            NaN                    NaN      NaN      NaN   
2 1914-03-01  6.0            NaN                    NaN      NaN      NaN   
3 1914-04-01  6.0            NaN                    NaN      NaN      NaN   
4 1914-05-01  5.9            NaN                    NaN      NaN      NaN   

   Employment  Labour_Force  Participation_Rate  Unemployment_Rate  \
0         NaN           NaN                 NaN                NaN   
1         NaN           NaN                 NaN                NaN   
2         NaN           NaN                 NaN                NaN   
3         NaN           NaN                 NaN                NaN   
4         NaN           NaN                 NaN                NaN   

   BCPI_Total  BCPI_NonEnergy  BCPI_Energy  BCPI_Metals  BCPI_Forestry  \
0         NaN             NaN  

In [9]:
print("Shape:", monthly.shape)

print("\nDate range:")
print(monthly["Date"].min(), "to", monthly["Date"].max())

print("\nDuplicate dates:")
print(monthly["Date"].duplicated().sum())

print("\nDuplicate rows:")
print(monthly.duplicated().sum())

print("\nMissing values:")
print(monthly.isna().sum().sort_values(ascending=False))

print("\nData types:")
print(monthly.dtypes)

Shape: (1354, 18)

Date range:
1914-01-01 00:00:00 to 2026-10-01 00:00:00

Duplicate dates:
0

Duplicate rows:
0

Missing values:
GDP_YoY                  1010
GDP_MoM                   999
GDP_Real_Millions_CAD     998
Labour_Force              744
Participation_Rate        744
Employment                744
Unemployment_Rate         744
BCPI_NonEnergy            696
BCPI_Forestry             696
BCPI_Total                696
BCPI_Fish                 696
BCPI_Agriculture          696
BCPI_Energy               696
BCPI_Metals               696
Inflation_YoY              13
CPI                         1
Date                        0
Availability_Date           0
dtype: int64

Data types:
Date                     datetime64[ns]
CPI                             float64
Inflation_YoY                   float64
GDP_Real_Millions_CAD           float64
GDP_MoM                         float64
GDP_YoY                         float64
Employment                      float64
Labour_Force            

In [21]:
monthly.to_csv(
    DATA_DIR / "canada_macro_monthly_combined.csv",
    index=False
)

In [10]:
expected_dates = pd.date_range(
    start=monthly["Date"].min(),
    end=monthly["Date"].max(),
    freq="MS"
)

print("Missing months:", len(
    expected_dates.difference(monthly["Date"])
))

Missing months: 0


In [11]:
monthly.loc[
    monthly["CPI"].isna(),
    ["Date", "CPI", "Inflation_YoY"]
]

,Date,CPI,Inflation_YoY
1353,2026-10-01,NaN,NaN


In [12]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data/processed")

stocks = pd.read_csv(DATA_DIR / "canada_stocks_cleaned.csv")
bank = pd.read_csv(DATA_DIR / "canada_bank_of_canada_cleaned.csv")

for df in [stocks, bank]:
    df["Date"] = pd.to_datetime(df["Date"])

In [13]:
bank = bank.rename(columns={
    "Date": "Bank_Availability_Date"
})

stocks = stocks.sort_values("Date").reset_index(drop=True)
bank = bank.sort_values("Bank_Availability_Date").reset_index(drop=True)

daily_data = pd.merge_asof(
    stocks,
    bank,
    left_on="Date",
    right_on="Bank_Availability_Date",
    direction="backward",
    tolerance=pd.Timedelta("7D")
)

In [14]:
monthly_available = monthly.copy()

monthly_available["Availability_Date"] = (
    monthly_available["Date"] + pd.DateOffset(months=3)
)

monthly_available = monthly_available.rename(columns={
    "Date": "Macro_Reference_Date",
    "Availability_Date": "Macro_Availability_Date"
})

monthly_available = monthly_available.sort_values(
    "Macro_Availability_Date"
)

In [15]:
daily_data = daily_data.sort_values("Date")

daily_data = pd.merge_asof(
    daily_data,
    monthly_available,
    left_on="Date",
    right_on="Macro_Availability_Date",
    direction="backward"
)

daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

In [16]:
print("Shape:", daily_data.shape)

print("\nDate range:")
print(daily_data["Date"].min(), "to", daily_data["Date"].max())

print("\nNumber of stocks:")
print(daily_data["Ticker"].nunique())

print("\nRows per ticker:")
print(daily_data["Ticker"].value_counts().sort_index())

print("\nMissing values:")
print(
    daily_data.isna().sum()
    .sort_values(ascending=False)
)

Shape: (63574, 32)

Date range:
2000-01-04 00:00:00 to 2026-11-06 00:00:00

Number of stocks:
10

Rows per ticker:
Ticker
ABX.TO     6744
BMO.TO     6744
BN.TO      6744
CNQ.TO     6744
CNR.TO     6744
ENB.TO     6744
RY.TO      6744
SHOP.TO    2878
SU.TO      6744
TD.TO      6744
Name: count, dtype: int64

Missing values:
USD_CAD                    38984
Policy_Rate                21090
Bond_Yield_10Y              2473
Bank_Availability_Date      2259
Date                           0
Ticker                         0
Open                           0
High                           0
Volume                         0
Adj Close                      0
Close                          0
Low                            0
Dividends                      0
Stock Splits                   0
Macro_Reference_Date           0
CPI                            0
Inflation_YoY                  0
GDP_Real_Millions_CAD          0
GDP_MoM                        0
GDP_YoY                        0
Employment     

In [17]:
print(
    "Duplicate Date-Ticker:",
    daily_data.duplicated(["Date", "Ticker"]).sum()
)

Duplicate Date-Ticker: 0


In [18]:
bank_columns = [
    "Policy_Rate",
    "Bond_Yield_10Y",
    "USD_CAD"
]

for col in bank_columns:
    print(f"\n{'=' * 40}")
    print("Column:", col)

    missing = daily_data[daily_data[col].isna()]

    print("Missing rows:", len(missing))

    print("First missing date:",
          missing["Date"].min())

    print("Last missing date:",
          missing["Date"].max())

    print("Unique missing dates:",
          missing["Date"].nunique())


Column: Policy_Rate
Missing rows: 21090
First missing date: 2000-01-04 00:00:00
Last missing date: 2026-11-06 00:00:00
Unique missing dates: 2343

Column: Bond_Yield_10Y
Missing rows: 2473
First missing date: 2000-01-04 00:00:00
Last missing date: 2026-11-06 00:00:00
Unique missing dates: 273

Column: USD_CAD
Missing rows: 38984
First missing date: 2000-01-04 00:00:00
Last missing date: 2026-09-30 00:00:00
Unique missing dates: 4285


In [19]:
for col in bank_columns:
    available = daily_data[daily_data[col].notna()]

    print(
        col,
        "| First:", available["Date"].min(),
        "| Last:", available["Date"].max()
    )

Policy_Rate | First: 2009-04-21 00:00:00 | Last: 2026-11-05 00:00:00
Bond_Yield_10Y | First: 2001-01-02 00:00:00 | Last: 2026-11-05 00:00:00
USD_CAD | First: 2017-01-03 00:00:00 | Last: 2026-11-06 00:00:00


In [20]:
daily_data.to_csv(
    DATA_DIR / "canada_integrated_daily.csv",
    index=False
)

# Data Integration – Summary
Combined six prepared datasets into a unified Canadian financial and macroeconomic dataset.
The integration process involved two main steps:

    1. Merging CPI, GDP, Labour Market, and BCPI into a monthly macroeconomic dataset using outer joins.
    
    2. Combining daily stock prices with Bank of Canada indicators and monthly macroeconomic features using backward merge_asof() operations.
    
A temporary three-month availability delay was applied to monthly indicators, while a seven-day tolerance was used for daily Bank of Canada data.
The final dataset contains 63,574 rows and 32 columns, covering ten TSX-listed companies from January 2000 to November 2026.
No duplicate Date-Ticker records were found. Missing values remain in some daily financial indicators due to differences in historical coverage and observation availability.
Result: A unified daily dataset combining Canadian stock market data with macroeconomic and financial indicators, ready for Feature Engineering and subsequent modeling.
The assumed macroeconomic publication delays will require further validation before final backtesting.
